# Sequential Probabilistic Inference & the Six Steps

*Course 2 — Linear Kalman Filter, Part 1. Derives the **generic Gaussian sequential-probabilistic-inference (SPI)** recursion that every Kalman filter (linear, extended, sigma-point) specializes. Builds on [04 · Random Variables](04_Random_Variables.ipynb) and [06 · Propagating Uncertainty](06_Stochastic_Processes_and_Propagating_Uncertainty.ipynb).*

**Style:** every equation gets a plain-language paraphrase (→); extra intuition is flagged **→ Intuition**.

### 🧩 The Model Framework

- We start from a general (possibly nonlinear) discrete-time state-space model:

$$
x_k = f(x_{k-1}, u_{k-1}, w_{k-1}), \qquad z_k = h(x_k, u_k, v_k).
$$

  → The state evolves from the previous state, known input, and process noise; the measurement is some function of the current state, input, and sensor noise.

- Goal: estimate $x_k$ using **all measurements up to now**, collected in the set

$$
\mathbb{Z}_k = \{z_0, z_1, \dots, z_k\}.
$$

  → Everything the filter is allowed to "know" at time $k$ is $\mathbb{Z}_k$. The estimate must be a function of this set only.

- **→ Intuition:** the development here is *fully general* — we specialize to a linear model in [08](08_Deriving_the_Linear_Kalman_Filter.ipynb) and reuse the exact same skeleton for nonlinear filters in Course 3.

### 🧩 Notation (the whole course lives on this)

- Superscript **"$-$"** = a **predicted** quantity, using only *past* measurements ($\mathbb{Z}_{k-1}$).
- Superscript **"$+$"** = an **estimated** quantity, using *past and present* ($\mathbb{Z}_k$).
- Hat **"$\hat{\ }$"** = a predicted or estimated value, e.g. $\hat{x}_k^-,\ \hat{x}_k^+$.
- Tilde **"$\tilde{\ }$"** = an **error**, truth minus prediction/estimate: $\tilde{x} = x - \hat{x}$.
- **$\Sigma$** with a subscript = a correlation/covariance; e.g. $\Sigma_{\tilde{x},k}^- = \mathbb{E}[\tilde{x}_k^-(\tilde{x}_k^-)^T]$.

- One sampling interval looks like:

$$
\hat{x}_k^- \;\to\; \hat{x}_k^+ \;\;\Big|\;\; \hat{x}_{k+1}^- \;\to\; \hat{x}_{k+1}^+
$$

  → Each step: **predict** ($-$) using the model, then **correct** ($+$) using the new measurement, then roll to the next $k$.

### 🧩 The Cost Function: Minimum Mean-Squared Error

- We want the estimate that minimizes expected squared error, conditioned on the data:

$$
\hat{x}_k^{\text{MMSE}} = \arg\min_{\hat{x}_k}\ \mathbb{E}\!\big[\,\|x_k - \hat{x}_k\|_2^2 \;\big|\; \mathbb{Z}_k \big].
$$

  → Among all possible guesses, pick the one whose squared distance to the (unknown) truth is smallest **on average given what we've measured**.

- Differentiating the quadratic cost and setting it to zero (using $\frac{d}{dX}X^TAX = (A+A^T)X$) collapses to a beautifully simple answer:

$$
\hat{x}_k^{+} = \mathbb{E}[\,x_k \mid \mathbb{Z}_k\,].
$$

  → **The optimal MMSE estimate is just the conditional mean.** All the filtering work reduces to computing (or approximating) this expectation recursively.

- **→ Intuition:** "best guess = average of what's still possible, given the data." Everything else is machinery to compute that average cheaply, one time-step at a time.

### 🧩 Prediction Error, Innovation, and Why They're Zero-Mean

- Define the **prediction error** and **innovation**:

$$
\tilde{x}_k^- = x_k - \hat{x}_k^-, \qquad \tilde{z}_k = z_k - \hat{z}_k, \quad\text{where } \hat{x}_k^- = \mathbb{E}[x_k\mid\mathbb{Z}_{k-1}],\ \hat{z}_k = \mathbb{E}[z_k\mid\mathbb{Z}_{k-1}].
$$

  → The prediction error is "truth minus our forecast." The **innovation** $\tilde{z}_k$ is the part of the new measurement that was *not* already predictable — literally *what's new*.

- By iterated expectation, both are zero-mean: $\ \mathbb{E}[\tilde{x}_k^-]=0,\ \ \mathbb{E}[\tilde{z}_k]=0$.

  → On average our forecasts are unbiased; the surprise averages to nothing. If the innovation were consistently non-zero, the model would be wrong.

- The prediction/correct structure comes from expanding $\hat{x}_k^+ = \mathbb{E}[x_k\mid\mathbb{Z}_k]$:

$$
\hat{x}_k^{+} = \hat{x}_k^{-} + \mathbb{E}\!\big[\tilde{x}_k^- \mid z_k\big].
$$

  → The corrected estimate = the prediction **plus** whatever the new measurement tells us about the prediction error. This single line is the seed of every Kalman filter.

### 🧩 The Generic Conditional Gaussian (where the gain comes from)

- The remaining task is $\mathbb{E}[\tilde{x}_k^- \mid z_k]$. Solve it generically: for jointly-Gaussian vectors stacked as $X = \begin{bmatrix} x \\ z\end{bmatrix}$ with

$$
\mathbb{E}[X] = \begin{bmatrix}\bar{x}\\ \bar{z}\end{bmatrix}, \qquad \Sigma_{\tilde{X}} = \begin{bmatrix} \Sigma_{\tilde{x}} & \Sigma_{\tilde{x}\tilde{z}} \\ \Sigma_{\tilde{z}\tilde{x}} & \Sigma_{\tilde{z}}\end{bmatrix},
$$

  the conditional distribution $f(x\mid z)$ is Gaussian with

$$
\mathbb{E}[x\mid z] = \bar{x} + \Sigma_{\tilde{x}\tilde{z}}\,\Sigma_{\tilde{z}}^{-1}(z - \bar{z}), \qquad
\operatorname{cov}(x\mid z) = \Sigma_{\tilde{x}} - \Sigma_{\tilde{x}\tilde{z}}\,\Sigma_{\tilde{z}}^{-1}\Sigma_{\tilde{z}\tilde{x}}.
$$

  → **Observing $z$ shifts the mean of $x$** by "how $x$ and $z$ co-vary" times "how surprising $z$ is," and **shrinks the covariance** by exactly the information the correlation carries. (Derived via the Schur complement of $\Sigma_{\tilde X}$; see the algebra in the course, omitted here.)

- **→ Intuition:** the factor $\Sigma_{\tilde{x}\tilde{z}}\Sigma_{\tilde{z}}^{-1}$ is a signal-to-noise weighting — trust the measurement in proportion to how informative and how clean it is. This *is* the Kalman gain in disguise.

### 🧩 The State Update and the Gain $L_k$

- Apply the generic result with $x = \tilde{x}_k^-$ and $z = z_k$ (so $\bar x = \mathbb{E}[\tilde x_k^-]=0$, $\bar z = \hat z_k$). The correction becomes

$$
\hat{x}_k^{+} = \hat{x}_k^{-} + \underbrace{\Sigma_{\tilde{x}\tilde{z},k}\,\Sigma_{\tilde{z},k}^{-1}}_{L_k}\,\tilde{z}_k = \hat{x}_k^{-} + L_k\,(z_k - \hat{z}_k).
$$

  → Corrected estimate = prediction **+ gain × innovation**. The gain $L_k = \Sigma_{\tilde{x}\tilde{z},k}\,\Sigma_{\tilde{z},k}^{-1}$ converts "surprise in the measurement" into "correction of the state."

- The estimation-error covariance shrinks accordingly:

$$
\Sigma_{\tilde{x},k}^{+} = \Sigma_{\tilde{x},k}^{-} - L_k\,\Sigma_{\tilde{z},k}\,L_k^{T}.
$$

  → Measuring can only **reduce** uncertainty (we subtract a positive-semidefinite term). Two outputs: the state estimate $\hat{x}_k^+$ **and** its confidence $\Sigma_{\tilde{x},k}^+$.

### 🧩 The Six Steps of Generic Gaussian SPI

The recursion splits into **Prediction** (steps 1a–1c) and **Correction** (steps 2a–2c), run once per sample:

**Prediction**

$$
\textbf{1a: } \hat{x}_k^- = \mathbb{E}[x_k\mid\mathbb{Z}_{k-1}] = \mathbb{E}[f(x_{k-1},u_{k-1},w_{k-1})\mid\mathbb{Z}_{k-1}]
$$
$$
\textbf{1b: } \Sigma_{\tilde{x},k}^- = \mathbb{E}[(\tilde{x}_k^-)(\tilde{x}_k^-)^T]
$$
$$
\textbf{1c: } \hat{z}_k = \mathbb{E}[z_k\mid\mathbb{Z}_{k-1}] = \mathbb{E}[h(x_k,u_k,v_k)\mid\mathbb{Z}_{k-1}]
$$

**Correction**

$$
\textbf{2a: } L_k = \Sigma_{\tilde{x}\tilde{z},k}\,\Sigma_{\tilde{z},k}^{-1} \qquad
\textbf{2b: } \hat{x}_k^+ = \hat{x}_k^- + L_k(z_k-\hat{z}_k) \qquad
\textbf{2c: } \Sigma_{\tilde{x},k}^+ = \Sigma_{\tilde{x},k}^- - L_k\,\Sigma_{\tilde{z},k}\,L_k^T
$$

  → **1a–1c:** guess the state, its uncertainty, and the expected measurement using the model alone. **2a–2c:** form the gain, fold in the actual measurement, and tighten the uncertainty.

- **KEY POINT:** the estimator outputs $\hat{x}_k^+$ *and* $\Sigma_{\tilde{x},k}^+$; the truth lies (with high confidence) within $\hat{x}_k^+ \pm 3\sqrt{\operatorname{diag}(\Sigma_{\tilde{x},k}^+)}$.

- **→ Intuition:** this recursion is *model-agnostic*. The only hard parts are the expectations in 1a/1c and the covariances — which are trivial for a **linear** model (next notebook) and approximated for nonlinear ones (Course 3).

### 🧩 Summary

- The MMSE-optimal estimate is the **conditional mean** $\hat{x}_k^+ = \mathbb{E}[x_k\mid\mathbb{Z}_k]$.

- It factors into **predict then correct**: $\hat{x}_k^+ = \hat{x}_k^- + L_k(z_k-\hat{z}_k)$, with the **innovation** $\tilde z_k = z_k-\hat z_k$ carrying the new information.

- The **gain** $L_k = \Sigma_{\tilde x\tilde z,k}\Sigma_{\tilde z,k}^{-1}$ and covariance shrink $\Sigma^+ = \Sigma^- - L_k\Sigma_{\tilde z,k}L_k^T$ both drop out of the **generic conditional-Gaussian** result.

- Packaged as **six steps** (1a–1c predict, 2a–2c correct), this is the template every Kalman filter fills in.

---
*Next: [08 · Deriving the Linear Kalman Filter](08_Deriving_the_Linear_Kalman_Filter.ipynb).*